In [1]:
# 02_determinant_recovery.ipynb
# Determinant recovery: do board characteristics predict the oversight-consistency
# metrics in the pre-specified directions? H1 AI literacy lowers metamorphic
# violations; H2 diversity raises Stage-1 dispersion; H3 escalation authority,
# independence, diversity and CEO tenure raise the convergence index. OLS with
# standardized continuous predictors. Figure is greyscale; no captions embedded.
import numpy as np, pandas as pd
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8})
DATA = Path("..")/"data"; FIG = Path("..")/"results"/"figures"; TAB = Path("..")/"results"/"tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def save(fig, name):
    fig.savefig(FIG/f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG/f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

b = pd.read_csv(DATA/"board_analysis.csv")
b = b[b.n_resp >= 3].copy()
for c in ["ai_literacy", "diversity", "independence", "ceo_tenure_z", "risk_culture"]:
    b[c + "_s"] = (b[c] - b[c].mean()) / b[c].std()

models = {
    "H1_violation":   smf.ols("viol1 ~ ai_literacy_s + diversity_s + independence_s + escalation + ceo_tenure_z_s + risk_culture_s", b).fit(),
    "H2_dispersion":  smf.ols("disp1 ~ diversity_s + ai_literacy_s", b).fit(),
    "H3_convergence": smf.ols("conv_disp ~ escalation + independence_s + diversity_s + ceo_tenure_z_s + risk_culture_s", b).fit(),
}
rows = []
for name, m in models.items():
    ci = m.conf_int()
    for term in m.params.index:
        rows.append(dict(model=name, term=term, coef=m.params[term], se=m.bse[term],
                         ci_lo=ci.loc[term, 0], ci_hi=ci.loc[term, 1], p=m.pvalues[term]))
tab = pd.DataFrame(rows).round(4)
tab.to_csv(TAB/"table03_determinants.csv", index=False)

# Figure 3: coefficient plot for the convergence model (H3)
m3 = models["H3_convergence"]
terms = [t for t in m3.params.index if t != "Intercept"]
ci = m3.conf_int().loc[terms]; est = m3.params[terms]
fig, ax = plt.subplots(figsize=(5.4, 3.8)); yy = np.arange(len(terms))[::-1]
ax.errorbar(est.values, yy, xerr=[est.values - ci[0].values, ci[1].values - est.values],
            fmt="s", color="#222222", ecolor="#777777", capsize=3, markeredgecolor="black")
ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_yticks(yy); ax.set_yticklabels(terms)
ax.set_xlabel("Standardized coefficient (convergence index)")
save(fig, "fig03_coef_convergence")

print("R2:  H1=%.3f  H2=%.3f  H3=%.3f" % (models["H1_violation"].rsquared,
      models["H2_dispersion"].rsquared, models["H3_convergence"].rsquared))
print(tab[tab.term != "Intercept"][["model", "term", "coef", "p"]].to_string(index=False))
print("figure: fig03  | table: table03")


R2:  H1=0.480  H2=0.742  H3=0.731
         model           term    coef      p
  H1_violation  ai_literacy_s -0.0604 0.0000
  H1_violation    diversity_s  0.0241 0.0009
  H1_violation independence_s  0.0046 0.5063
  H1_violation     escalation -0.0075 0.5925
  H1_violation ceo_tenure_z_s  0.0098 0.1749
  H1_violation risk_culture_s -0.0102 0.1480
 H2_dispersion    diversity_s  0.1099 0.0000
 H2_dispersion  ai_literacy_s -0.2529 0.0000
H3_convergence     escalation  0.1850 0.0000
H3_convergence independence_s  0.0826 0.0000
H3_convergence    diversity_s  0.0707 0.0000
H3_convergence ceo_tenure_z_s  0.1036 0.0000
H3_convergence risk_culture_s  0.0389 0.0006
figure: fig03  | table: table03
